In [9]:
import ollama

file_path = 'myd.md'
with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()
paragraphs = content.split('##')

for i,paragraph in enumerate(paragraphs):
    print(f'paragraph {i + 1}:\n{paragraph}\n')
    print('-' * 20)

paragraph 1:
---
title: 'FAQ'
category: 'faq'
---

# 常见问题



--------------------
paragraph 2:
 1. openGauss 数据库开源许可协议是什么？

木兰宽松许可证 MulanPSL2 V2，无传染



--------------------
paragraph 3:
 2. openGauss 支持部署的硬件架构及操作系统有哪些？

ARM：openEuler 20.03LTS（推荐采用此操作系统）｜ openEuler 22.03LTS ｜ Kylin-V10 ｜ FusionOS 22 ｜统信

X86：openEuler 20.03LTS ｜ openEuler 22.03LTS ｜ Kylin-V10 ｜ CentOS 7.6 ｜ Asianux 7.6 ｜ FusionOS 22 ｜统信

<br/>

ubuntu / centos8 /centos10 / 红旗 需要适配编译数据库；在飞腾/海光 等服务器上安装需要重新适配编译。



--------------------
paragraph 4:
 3. openGauss 有哪些版本？

openGauss 社区每两年发布一个 LTS 版本，LTS 版本作为长期支持版本，可规模上线使用。半年发布一个创新版本，创新版本供用户联创测试使用；涉及重大问题修复时，会按需发布补丁版本。同时按照不同场景分为以下版本：

1. openGauss 企业版:具备更齐全的集群管理功能,适合企业用户；
2. openGauss 极简版:安装配置简单,解压可用,适合个人开发者；
3. openGauss 轻量版:精简功能,缩减安装包大小,内存占用更少；
4. openGauss 分布式镜像:基于 ShardingSphere 和 k8s 的分布式容器化镜像。

详情参考 openGauss 官网[“学习”->“文档”](https://docs.opengauss.org)区域。



--------------------
paragraph 5:
 4. openGauss 分布式部署方案是什么？

1. 基于 openLookeng 实现分布式分析能力，与 shardingsphere 配合 openGau

In [10]:
def embedding(text):
    vector = ollama.embeddings(model="nomic-embed-text", prompt=text)
    return vector["embedding"]

In [11]:
text = "openGauss 是一款开源数据库"
emb = embedding(text)
dimensions = len(emb)

print("text: {}, embedding dim : {}, embedding : {}...".format(text, dimensions, emb[:10]))


text: openGauss 是一款开源数据库, embedding dim : 768, embedding : [-0.5392146110534668, 1.3381578922271729, -3.5275259017944336, -1.0050768852233887, -0.19613319635391235, 0.2840339243412018, -0.4675346612930298, 0.08503472805023193, -0.22875681519508362, -0.9965018033981323]...


In [12]:
import psycopg2

table_name = "opengauss_data"
conn = psycopg2.connect(
    database = "db1",
    user="user1",
    password="Test@123",
    host="127.0.0.1",
    port="5432"
)

cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS {};".format(table_name))
cur.execute("CREATE TABLE {} (id INT PRIMARY KEY, content TEXT, emb vector({}))".format(table_name,dimensions))
conn.commit()

In [13]:
for i,paragraph in enumerate(paragraphs):
    emb = embedding(paragraph)
    insert_data_sql = f'''INSERT INTO {table_name} (id, content, emb) VALUES (%s, %s, %s);'''
    cur.execute(insert_data_sql, (i, paragraph, emb))
    conn.commit()

cur.execute("CREATE INDEX ON {} USING HNSW (emb vector_l2_ops);".format(table_name))
conn.commit()

In [14]:
cur.execute("SELECT * FROM {};".format(table_name))
data_vector = cur.fetchall()
print("向量表的数据:", data_vector)

file_path = 'result.txt'
with open(file_path, "w", encoding="utf-8") as f:
    for t in data_vector:
        f.write(str(t) + "\n")

向量表的数据: [(0, "---\ntitle: 'FAQ'\ncategory: 'faq'\n---\n\n# 常见问题\n\n", '[0.34152254,1.6058762,-3.6505022,-1.0067232,1.4491947,0.019693762,-0.0019339789,-0.6326871,-0.40344885,-1.2108005,-0.10451098,0.30761507,0.8156114,1.1429014,0.5308527,-0.8476516,-0.3929885,-1.2558231,-1.3968388,1.0421475,-0.23169911,-0.056122728,-0.87705505,-0.042974044,1.7609072,0.5397763,0.6031816,0.34938735,-1.010622,0.4780243,-0.539583,0.14034057,-1.098949,0.4313456,-0.19846381,-0.16551907,1.108882,0.6914598,-0.44484603,0.24399881,1.7809014,-0.07479064,0.31105715,-0.6209339,0.64523005,-0.35267887,0.87238073,0.5975125,1.7199107,-0.24949609,-0.6884125,0.10661705,1.5997541,-0.2062765,1.3414272,-0.04857196,0.75613976,1.2163827,-0.4594929,1.2054328,0.8482686,0.8288926,-0.58442676,1.139617,0.5602176,-1.3645504,0.3954197,2.0158026,-1.3240441,-0.29973555,0.26552856,0.024936661,0.24712634,0.97444606,-0.87077975,-0.5659379,-0.2148232,0.2275205,0.03781127,-0.12237069,-0.658289,-0.04339484,1.3983258,-0.22344151,0.66380286,-

In [15]:
question = "openGauss 发布了哪些版本？"

emb_data = embedding(question)
dimensions = len(emb_data)

cur = conn.cursor()
cur.execute("select content from {} order by emb <-> '{}' limit 1;".format(table_name, emb_data))
conn.commit()

rows = cur.fetchall()
print(rows)

cur.close()
conn.close()

[(' 3. openGauss 有哪些版本？\n\nopenGauss 社区每两年发布一个 LTS 版本，LTS 版本作为长期支持版本，可规模上线使用。半年发布一个创新版本，创新版本供用户联创测试使用；涉及重大问题修复时，会按需发布补丁版本。同时按照不同场景分为以下版本：\n\n1. openGauss 企业版:具备更齐全的集群管理功能,适合企业用户；\n2. openGauss 极简版:安装配置简单,解压可用,适合个人开发者；\n3. openGauss 轻量版:精简功能,缩减安装包大小,内存占用更少；\n4. openGauss 分布式镜像:基于 ShardingSphere 和 k8s 的分布式容器化镜像。\n\n详情参考 openGauss 官网[“学习”->“文档”](https://docs.opengauss.org)区域。\n\n',)]


In [16]:
context = "\n".join(row[0] for row in rows)
SYSTEM_PROMPT = "你作为一个对话 AI 助手，结合上下文信息简练高效地回答用户提出的问题"
USER_PROMPT = f"请结合{context}信息来回答{question}的问题，不需要额外的无用回答"


response = ollama.chat(
    model = "deepseek-r1",
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
)
print(response["message"]["content"])

<think>
好，用户问的是“openGauss 发布了哪些版本？”根据提供的上下文，我知道openGauss每年发布两个主要版本：LTS和创新版。LTS是长期支持版本，半年发布一次创新版供用户联创测试。此外，还有企业版、极简版、轻量版和分布式镜像这些特定类型的版本。

我需要把这些信息整合起来，简洁明了地回答用户的问题。不需要额外的内容，只提供关键点即可。这样用户就能清楚了解openGauss有哪些不同的发布版本，并且知道每种版本适合什么样的用户群体。
</think>

openGauss 每年发布两个主要版本：

1. **LTS 版本**：作为长期支持版本，半年发布一次。
2. **创新版**：供用户联创测试。

此外，按场景分为：
- 企业版：适合企业用户
- 极简版：安装配置简单，适合个人开发者
- 轻量版：精简功能，缩减安装包大小
- 分布式镜像：基于 ShardingSphere 和 k8s 的分布式容器化镜像

详情可参考 openGauss 官网。


In [17]:
USER_PROMPT = f"请回答{question}的问题，不需要额外的无用回答"


response = ollama.chat(
    model = "deepseek-r1",
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
)
print(response["message"]["content"])


<think>
好的，我需要回答关于openGauss发布版本的问题。首先，我应该收集最新的发布信息。通常，开放 Gauss 定期发布新版本，每次都有新的功能和改进。

接下来，我会查找最近几个版本的具体发布日期和主要更新内容。例如，可能包括g trunk、g dev、g beta等不同的版本类型。

然后，我要确保信息准确，并且回答简明扼要，避免包含任何额外的内容或分页提示。

最后，我将这些信息整理成一个清晰易懂的回答，让用户能够快速了解openGauss当前的发布情况。
</think>

openGauss 最新版本包括 g trunk、g dev 和 g beta 等不同阶段的发布。具体的版本号和发布时间需要参考官方发布的详细信息。
